# HVFHV - Feature Selection & Engineering

Feature-elimination analysis for the HVFHV track: variance / degeneracy check, correlation pruning, multicollinearity (VIF), burden-metric redundancy, and leakage classification. Decisions are logged to `results/hvfhv_feature_selection.csv` and copied to `results/feature_selection.csv` for the generic project todo.

**Scope.** This is a descriptive inference project, not a demand-prediction task. Feature selection means removing redundant, degenerate, leaky, or unreliable variables while preserving the design-selected burden, exposure, and pre-policy control features.

**Data.** This notebook uses only small project-facing files:

- `data/processed/samples/hvfhv_trip_level_sample_eda_features.csv`
- `data/processed/modeling/hvfhv_model1_zone_features.csv`
- `data/processed/disruption_score/hvfhv_monthly_panel.csv`

It does not read raw files or full parquet files, and it does not rebuild DS_z or Model 2.

**How to read this.** Part A confirms that design-selected features are usable: not degenerate, not redundant except where intentionally engineered, and leakage-clean for the intended model role. Part B documents engineered features already produced upstream.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)

# Resolve repo root robustly, whether the notebook runs from repo root or notebooks/.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "data" / "processed").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

SAMPLE = REPO_ROOT / "data" / "processed" / "samples" / "hvfhv_trip_level_sample_eda_features.csv"
MODEL1 = REPO_ROOT / "data" / "processed" / "modeling" / "hvfhv_model1_zone_features.csv"
PANEL = REPO_ROOT / "data" / "processed" / "disruption_score" / "hvfhv_monthly_panel.csv"
RESULTS = REPO_ROOT / "results" / "hvfhv_feature_selection.csv"
TODO_RESULTS = REPO_ROOT / "results" / "feature_selection.csv"
RESULTS.parent.mkdir(parents=True, exist_ok=True)

for path in [SAMPLE, MODEL1, PANEL]:
    if not path.exists():
        raise FileNotFoundError(path)

trips = pd.read_csv(SAMPLE)
model1 = pd.read_csv(MODEL1)
panel = pd.read_csv(PANEL)

# Numeric weekday code for variance/correlation screening; keep original label for reporting.
if "pickup_weekday" in trips.columns and "pickup_weekday_code" not in trips.columns:
    trips["pickup_weekday_code"] = pd.Categorical(trips["pickup_weekday"]).codes

# Normalize boolean-like columns for numeric screening.
for flag_col in ["charged_cbd_flag", "shared_request_yes_flag", "shared_match_yes_flag", "main_analysis_flag", "burden_analysis_flag"]:
    if flag_col in trips.columns:
        if trips[flag_col].dtype == object:
            trips[flag_col] = trips[flag_col].map({"True": True, "False": False, "true": True, "false": False, "Y": True, "N": False}).fillna(trips[flag_col])
        trips[flag_col] = trips[flag_col].astype(bool)

def show(frame, max_rows=30):
    if isinstance(frame, pd.Series):
        print(frame.to_string())
    elif isinstance(frame, pd.DataFrame):
        if len(frame) > max_rows:
            print(frame.head(max_rows).to_string(index=False))
            print(f"... {len(frame) - max_rows} more rows")
        else:
            print(frame.to_string(index=False))
    else:
        print(frame)

print(f"HVFHV EDA-feature sample rows: {len(trips):,}")
print(f"Model 1 zone-direction rows: {len(model1):,}")
print(f"Model 2 monthly panel rows: {len(panel):,}")
print("Sample rows by year:")
show(trips.groupby("year").size().rename("rows"))

def spearman_pair(left, right):
    pair = pd.DataFrame({"left": pd.to_numeric(left, errors="coerce"), "right": pd.to_numeric(right, errors="coerce")}).dropna()
    if len(pair) < 2:
        return np.nan
    return pair["left"].rank(method="average").corr(pair["right"].rank(method="average"))


HVFHV EDA-feature sample rows: 16,889
Model 1 zone-direction rows: 525
Model 2 monthly panel rows: 5,236
Sample rows by year:
year
2024    8473
2025    8416


## 1. Variance / degeneracy check

A near-constant feature carries no useful screening information. The policy fields are expected to be constant in 2024 because the CBD fee was not yet in effect; that is a design fact, not a data defect.

In [2]:
variance_candidates = {
    "trip_distance_miles": "control candidate",
    "trip_duration_minutes": "redundancy check",
    "passenger_cost_pretip": "cost outcome / denominator basis",
    "base_passenger_fare": "baseline fare component",
    "driver_pay": "driver-side descriptive outcome",
    "pickup_hour": "temporal EDA-only",
    "pickup_weekday_code": "temporal EDA-only",
    "shared_request_yes_flag": "shared-ride regime EDA-only",
    "shared_match_yes_flag": "shared-ride regime EDA-only",
    "cbd_congestion_fee": "policy numerator",
    "charged_cbd_flag": "post-policy observed charge flag",
}

rows = []
for feature, role in variance_candidates.items():
    series = pd.to_numeric(trips[feature], errors="coerce")
    by_year = trips.groupby("year")[feature].apply(lambda x: pd.to_numeric(x, errors="coerce").std(ddof=0))
    std_2024 = by_year.get(2024, np.nan)
    std_2025 = by_year.get(2025, np.nan)
    rows.append({
        "feature": feature,
        "role": role,
        "std_overall": series.std(ddof=0),
        "std_2024": std_2024,
        "std_2025": std_2025,
        "near_constant_overall": bool(series.nunique(dropna=True) <= 1),
        "near_constant_2024": bool(pd.notna(std_2024) and std_2024 == 0),
    })

var_tbl = pd.DataFrame(rows)
show(var_tbl.round(4), max_rows=40)
print()
print("Observed charged flag mean by year:")
show(trips.groupby("year")["charged_cbd_flag"].mean().round(4))

                feature                             role  std_overall  std_2024  std_2025  near_constant_overall  near_constant_2024
    trip_distance_miles                control candidate       5.7638    5.8268    5.6994                  False               False
  trip_duration_minutes                 redundancy check      14.1802   14.3679   13.9826                  False               False
  passenger_cost_pretip cost outcome / denominator basis      28.0371   27.7808   28.2827                  False               False
    base_passenger_fare          baseline fare component      23.1870   22.9490   23.4189                  False               False
             driver_pay  driver-side descriptive outcome      17.2557   17.0966   17.4126                  False               False
            pickup_hour                temporal EDA-only       6.4262    6.4745    6.3772                  False               False
    pickup_weekday_code                temporal EDA-only       1.9834

**Finding.** The trip-shape, cost, provider/regime, and time fields have usable variation in the HVFHV sample. `cbd_congestion_fee` and `charged_cbd_flag` are degenerate in 2024 by design, so they are not cross-year controls; geography-based exposure is used for Model 2 instead.

## 2. Correlation pruning (trip-economics candidates)

The candidate continuous trip-economics fields are screened for redundancy. Cost components are useful for reconstructing passenger cost, but highly correlated cost fields should not be treated as independent controls.

In [3]:
econ = [
    "trip_distance_miles",
    "trip_duration_minutes",
    "base_passenger_fare",
    "passenger_cost_pretip",
    "passenger_cost_excl_cbd",
    "base_cost_ex_cbd",
    "congestion_surcharge",
    "tolls",
    "airport_fee",
    "bcf",
    "sales_tax",
    "driver_pay",
]
sub = trips[econ].apply(pd.to_numeric, errors="coerce")
sub = sub[(sub["trip_distance_miles"] >= 0) & (sub["trip_distance_miles"] < 200)]
pear = sub.corr("pearson")
spear = sub.rank(method="average").corr("pearson")

print("Pearson r:")
show(pear.round(3), max_rows=20)
print()
print("Selected correlations (Pearson | Spearman):")
for column in ["trip_duration_minutes", "base_passenger_fare", "passenger_cost_pretip", "driver_pay"]:
    print(f"  distance ~ {column:24s} {pear.loc['trip_distance_miles', column]: .3f} | {spear.loc['trip_distance_miles', column]: .3f}")
print()
print(f"base_fare ~ passenger_cost_pretip = {pear.loc['base_passenger_fare', 'passenger_cost_pretip']:.3f}")
print(f"passenger_cost_pretip ~ passenger_cost_excl_cbd = {pear.loc['passenger_cost_pretip', 'passenger_cost_excl_cbd']:.3f}")
print(f"passenger_cost_pretip ~ base_cost_ex_cbd = {pear.loc['passenger_cost_pretip', 'base_cost_ex_cbd']:.3f}")

Pearson r:
 trip_distance_miles  trip_duration_minutes  base_passenger_fare  passenger_cost_pretip  passenger_cost_excl_cbd  base_cost_ex_cbd  congestion_surcharge  tolls  airport_fee   bcf  sales_tax  driver_pay
               1.000                  0.799                0.864                  0.862                    0.864             0.864                 0.029  0.541        0.439 0.855      0.615       0.923
               0.799                  1.000                0.817                  0.816                    0.817             0.817                 0.132  0.440        0.369 0.808      0.667       0.907
               0.864                  0.817                1.000                  0.992                    0.993             0.993                 0.176  0.517        0.459 0.976      0.740       0.936
               0.862                  0.816                0.992                  1.000                    1.000             1.000                 0.225  0.592        0.481 0.979   

**Finding.** The cost family is strongly collinear because the HVFHV passenger-cost reconstruction is a sum of mandatory components. Keep the reconstructed cost for outcomes and burden denominators, but avoid treating each component as an independent explanatory control.

## 3. Multicollinearity (VIF)

VIF quantifies how much each candidate can be explained by the others. VIF above 10 is a common warning threshold. This notebook computes VIF directly with NumPy so it can run without optional statsmodels dependencies.

In [4]:
def calculate_vif_table(frame, columns):
    clean = frame[columns].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().copy()
    clean = clean.loc[:, [col for col in clean.columns if clean[col].std(ddof=0) > 0]]
    standardized = (clean - clean.mean()) / clean.std(ddof=0)
    rows = []
    for column in standardized.columns:
        y = standardized[column].to_numpy()
        other_columns = [col for col in standardized.columns if col != column]
        X = standardized[other_columns].to_numpy()
        X = np.column_stack([np.ones(len(X)), X])
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        y_hat = X @ beta
        ss_res = float(np.sum((y - y_hat) ** 2))
        ss_tot = float(np.sum((y - y.mean()) ** 2))
        r_squared = 1.0 - ss_res / ss_tot if ss_tot else np.nan
        vif_value = np.inf if pd.notna(r_squared) and (1.0 - r_squared) <= 1e-12 else 1.0 / (1.0 - r_squared)
        rows.append({"feature": column, "VIF": vif_value, "n_rows": len(clean)})
    return pd.DataFrame(rows).sort_values("VIF", ascending=False)

vif_columns = [
    "trip_distance_miles",
    "trip_duration_minutes",
    "base_passenger_fare",
    "passenger_cost_pretip",
    "driver_pay",
]
vif_tbl = calculate_vif_table(sub, vif_columns)
show(vif_tbl.round(2))

              feature   VIF  n_rows
  base_passenger_fare 72.19   16889
passenger_cost_pretip 66.42   16889
           driver_pay 25.67   16889
  trip_distance_miles  7.17   16889
trip_duration_minutes  6.27   16889


**Finding.** Passenger cost, base fare, and driver pay are related dimensions of trip economics. Use at most one declared trip-economics control in a compact specification, and keep driver pay as descriptive outcome/context rather than a control for volume response.

## 4. Burden-metric redundancy: relative burden, DS_z, and base cost

On 2025 charged HVFHV trips with a base-cost floor, we check whether the relative-burden fields duplicate the trip-level DS_z calculation and why same-year 2025 cost variables are not clean controls.

In [5]:
charged = trips[(trips["year"] == 2025) & (trips["charged_cbd_flag"])].copy()
charged["base_cost_for_ds"] = pd.to_numeric(charged["base_cost_ex_cbd"], errors="coerce").round(2)
charged = charged[charged["base_cost_for_ds"] >= 1.0].copy()
charged["ds_z_trip"] = charged["cbd_congestion_fee"] / charged["base_cost_for_ds"]
charged["inv_base_cost"] = 1.0 / charged["base_cost_for_ds"]
charged["inv_distance"] = 1.0 / charged["trip_distance_miles"].replace(0, np.nan)

print(f"n (2025 charged HVFHV, base >= $1): {len(charged):,}")
print(f"corr(relative base-cost burden, DS_z) Pearson = {charged['relative_cbd_burden_base_cost'].corr(charged['ds_z_trip']):.4f}")
print(f"corr(relative current-cost burden, DS_z) Pearson = {charged['relative_cbd_burden_current_cost'].corr(charged['ds_z_trip']):.4f}")
print(f"corr(DS_z, 1/base_cost) Pearson = {charged['ds_z_trip'].corr(charged['inv_base_cost']):.4f}")
print(f"corr(DS_z, base_cost) Spearman = {spearman_pair(charged['ds_z_trip'], charged['base_cost_for_ds']):.4f}")
print(f"corr(DS_z, distance) Spearman = {spearman_pair(charged['ds_z_trip'], charged['trip_distance_miles']):.4f}")
print(f"corr(DS_z, duration) Spearman = {spearman_pair(charged['ds_z_trip'], charged['trip_duration_minutes']):.4f}")

n (2025 charged HVFHV, base >= $1): 2,954
corr(relative base-cost burden, DS_z) Pearson = 1.0000
corr(relative current-cost burden, DS_z) Pearson = 0.9994
corr(DS_z, 1/base_cost) Pearson = 1.0000
corr(DS_z, base_cost) Spearman = -1.0000
corr(DS_z, distance) Spearman = -0.8193
corr(DS_z, duration) Spearman = -0.8190


**Finding.** The base-cost burden field is mechanically the trip-level DS_z quantity. `relative_cbd_burden_*` is useful as a reference check, but it should not be included as an independent feature alongside DS_z.

## 5. Model-level leakage and exposure checks

The feature table and monthly panel already contain the modeling units used downstream. Here we verify which fields are outcomes, which are pre-policy controls, and whether Model 2 exposure is present and bounded.

In [6]:
zone_numeric_cols = [
    "DS_z",
    "DS_z_median",
    "pct_volume_change",
    "n_trips_2024",
    "avg_base_fare_2024",
    "avg_total_cost_2024",
    "avg_trip_distance_2024",
    "avg_trip_duration_2024",
    "log_n_trips_2024",
]
zone_numeric = model1[zone_numeric_cols].apply(pd.to_numeric, errors="coerce")
zone_corr = zone_numeric.corr("pearson")
print("Model 1 feature/outcome Pearson r:")
show(zone_corr.round(3), max_rows=20)

unit_exposure = panel[["zone", "direction", "charged_share_2024_geo"]].drop_duplicates()
print()
print("Model 2 exposure QA:")
print(f"panel rows: {len(panel):,}")
print(f"zone-direction units: {len(unit_exposure):,}")
print(f"missing exposure rows: {int(panel['charged_share_2024_geo'].isna().sum()):,}")
print(f"exposure min/max: {panel['charged_share_2024_geo'].min():.4f} / {panel['charged_share_2024_geo'].max():.4f}")
print()
print("Exposure distribution by unit:")
show(unit_exposure["charged_share_2024_geo"].describe().round(4))

Model 1 feature/outcome Pearson r:
  DS_z  DS_z_median  pct_volume_change  n_trips_2024  avg_base_fare_2024  avg_total_cost_2024  avg_trip_distance_2024  avg_trip_duration_2024  log_n_trips_2024
 1.000        0.997             -0.525         0.406               0.119                0.178                  -0.301                   0.197             0.408
 0.997        1.000             -0.521         0.404               0.092                0.149                  -0.319                   0.177             0.421
-0.525       -0.521              1.000        -0.281              -0.010               -0.011                   0.115                  -0.075            -0.447
 0.406        0.404             -0.281         1.000               0.348                0.368                   0.142                   0.381             0.632
 0.119        0.092             -0.010         0.348               1.000                0.989                   0.744                   0.803            -0.192
 0.17

**Finding.** `pct_volume_change`, `n_trips_2025`, `delta_volume`, monthly `n_trips`, and `log_n_trips` are outcomes or outcome ingredients. Model 2 uses `charged_share_2024_geo` as the pre-policy geography-based exposure; observed 2025 charge status remains diagnostic only.

## 6. Decision table -> `results/hvfhv_feature_selection.csv`

Each candidate feature is logged with computed screening metrics where available and a keep, drop, engineer, primary, outcome, reference, or EDA-only decision.

In [7]:
max_abs_corr = {}
for column in econ:
    off_diagonal = pear[column].drop(index=column).abs()
    max_abs_corr[column] = off_diagonal.max()

zone_max_abs_corr = {}
for column in zone_numeric.columns:
    off_diagonal = zone_corr[column].drop(index=column).abs()
    zone_max_abs_corr[column] = off_diagonal.max()

vif_lookup = dict(zip(vif_tbl["feature"], vif_tbl["VIF"]))
near_const_2024 = dict(zip(var_tbl["feature"], var_tbl["near_constant_2024"]))

def near_constant_value(feature):
    if feature in near_const_2024:
        return near_const_2024[feature]
    if feature in model1.columns:
        return bool(model1[feature].nunique(dropna=True) <= 1)
    if feature in panel.columns:
        return bool(panel.loc[panel["year"] == 2024, feature].nunique(dropna=True) <= 1)
    return ""

def max_corr_value(feature):
    if feature in max_abs_corr:
        return round(float(max_abs_corr[feature]), 3)
    if feature in zone_max_abs_corr:
        return round(float(zone_max_abs_corr[feature]), 3)
    return ""

def vif_value(feature):
    if feature in vif_lookup:
        return round(float(vif_lookup[feature]), 2)
    return ""

spec = [
    ("pickup_hour", "trip", "EDA-only", "safe_2024", "descriptive temporal EDA; not a model feature"),
    ("pickup_weekday", "trip", "EDA-only", "safe_2024", "descriptive temporal EDA; not a model feature"),
    ("PULocationID", "trip", "engineer", "safe_2024", "defines pickup zone-direction unit and CRZ geography exposure"),
    ("DOLocationID", "trip", "engineer", "safe_2024", "defines dropoff zone-direction unit and CRZ geography exposure"),
    ("hvfhs_license_num", "trip", "EDA-only", "context", "provider mix context; not a primary causal control"),
    ("provider_label", "trip", "EDA-only", "context", "provider mix context; use descriptively"),
    ("shared_request_yes_flag", "trip", "EDA-only", "context", "shared-ride regime context; not a primary model feature"),
    ("shared_match_yes_flag", "trip", "EDA-only", "context", "shared-ride regime context; not a primary model feature"),
    ("trip_distance_miles", "trip", "keep", "safe_2024", "physical trip-shape control candidate when aggregated from pre-policy data"),
    ("trip_duration_minutes", "trip", "drop", "redundant", "correlated with distance and potentially affected by traffic conditions"),
    ("base_passenger_fare", "trip", "context", "safe_2024", "component of reconstructed passenger cost; use as 2024 aggregate sensitivity only"),
    ("passenger_cost_pretip", "trip", "keep", "outcome", "cost outcome and DS_z denominator basis; not a 2025 control"),
    ("passenger_cost_excl_cbd", "trip", "engineer", "redundant", "cost denominator precursor; near-duplicate of passenger cost outside fee adjustment"),
    ("base_cost_ex_cbd", "trip", "engineer", "redundant", "rounded DS_z denominator with $1 floor for burden analysis"),
    ("cbd_congestion_fee", "trip", "keep", "cleaning", "policy fee and DS_z numerator; degenerate in 2024 by design"),
    ("charged_cbd_flag", "trip", "replace", "forbidden_2025", "observed post-policy charge flag; use geography exposure for cross-year Model 2"),
    ("relative_cbd_burden_current_cost", "trip", "reference", "redundant", "reference burden metric; not independent from DS_z"),
    ("relative_cbd_burden_base_cost", "trip", "reference", "redundant", "trip-level DS_z equivalent; reference only"),
    ("congestion_surcharge", "trip", "context", "cleaning", "legacy surcharge component; not a standalone model feature"),
    ("tolls", "trip", "context", "cleaning", "passenger-cost component; not a standalone model feature"),
    ("airport_fee", "trip", "context", "safe_2024", "airport exposure context; not a primary model feature"),
    ("bcf", "trip", "context", "cleaning", "passenger-cost component; not a standalone model feature"),
    ("sales_tax", "trip", "context", "cleaning", "passenger-cost component; not a standalone model feature"),
    ("tips", "trip", "drop", "outcome", "post-policy rider payment behavior; not used as volume-control feature"),
    ("driver_pay", "trip", "EDA-only", "outcome", "driver-side outcome/context; not a post-policy volume-control feature"),
    ("DS_z", "zone", "primary", "outcome", "Model 1 burden metric; 2025 charged trips with $1 denominator floor"),
    ("DS_z_median", "zone", "reference", "outcome", "robustness companion to mean DS_z"),
    ("pct_volume_change", "zone", "outcome", "outcome", "Model 1 outcome; never use as a predictor of itself"),
    ("delta_volume", "zone", "outcome", "outcome", "outcome ingredient; not an explanatory control"),
    ("n_trips_2024", "zone", "keep", "safe_2024", "pre-policy baseline volume control"),
    ("n_trips_2025", "zone", "drop", "forbidden_2025", "post-policy outcome ingredient"),
    ("avg_base_fare_2024", "zone", "keep", "safe_2024", "pre-policy fare sensitivity; use carefully with burden"),
    ("avg_total_cost_2024", "zone", "keep", "safe_2024", "pre-policy cost sensitivity; use carefully with burden"),
    ("avg_trip_distance_2024", "zone", "keep", "safe_2024", "pre-policy trip-shape control"),
    ("avg_trip_duration_2024", "zone", "sensitivity", "safe_2024", "pre-policy duration sensitivity; interpret cautiously"),
    ("Borough", "zone", "keep", "safe_2024", "geographic context and robustness grouping"),
    ("charged_share_2024_geo", "panel", "primary", "safe_2024", "Model 2 pre-policy geography-based exposure"),
    ("post", "panel", "keep", "design", "post-period indicator for Model 2 interaction"),
    ("n_trips", "panel", "outcome", "outcome", "monthly trip-count outcome ingredient"),
    ("log_n_trips", "panel", "outcome", "outcome", "Model 2 outcome"),
    ("2025 observed charged share", "panel", "diagnostic", "forbidden_2025", "validation diagnostic only; not main exposure"),
]

feature_selection = pd.DataFrame(spec, columns=["feature", "level", "decision", "leakage_status", "reason"])
feature_selection["near_constant_2024"] = feature_selection["feature"].map(near_constant_value)
feature_selection["max_abs_corr_pearson"] = feature_selection["feature"].map(max_corr_value)
feature_selection["VIF"] = feature_selection["feature"].map(vif_value)
feature_selection = feature_selection[[
    "feature",
    "level",
    "near_constant_2024",
    "max_abs_corr_pearson",
    "VIF",
    "leakage_status",
    "decision",
    "reason",
]]
feature_selection.to_csv(RESULTS, index=False)
feature_selection.to_csv(TODO_RESULTS, index=False)
print(f"Wrote {RESULTS.relative_to(REPO_ROOT)} ({len(feature_selection)} features)")
print(f"Copied to {TODO_RESULTS.relative_to(REPO_ROOT)}")
show(feature_selection, max_rows=50)

Wrote results\hvfhv_feature_selection.csv (41 features)
Copied to results\feature_selection.csv
                         feature level near_constant_2024 max_abs_corr_pearson    VIF leakage_status    decision                                                                              reason
                     pickup_hour  trip              False                                  safe_2024    EDA-only                                       descriptive temporal EDA; not a model feature
                  pickup_weekday  trip                                                     safe_2024    EDA-only                                       descriptive temporal EDA; not a model feature
                    PULocationID  trip                                                     safe_2024    engineer                       defines pickup zone-direction unit and CRZ geography exposure
                    DOLocationID  trip                                                     safe_2024    engineer    

## 7. Summary

- Degenerate by design in 2024: `cbd_congestion_fee` and `charged_cbd_flag`. Keep them for cleaning, burden, and 2025 diagnostics, but do not use observed 2025 charge status as the main Model 2 exposure.
- Redundant cost family: `passenger_cost_pretip`, `passenger_cost_excl_cbd`, `base_cost_ex_cbd`, base fare, and components are tightly related. Use the reconstructed cost for outcomes and denominators; avoid stacking cost components as independent controls.
- Burden redundancy: `relative_cbd_burden_base_cost` is the trip-level DS_z quantity. Keep DS_z as the main burden metric and use relative burden fields only as references.
- Model 1: keep `DS_z`, baseline 2024 volume, and one declared 2024 trip-economics sensitivity at a time; `pct_volume_change` is the outcome.
- Model 2: keep `charged_share_2024_geo` as the main exposure and `log_n_trips` as the outcome. The negative estimate is suggestive association, estimated under assumptions, with a placebo warning from the no-June 2023-vs-2024 diagnostic.

# Part B - Feature Engineering

This notebook documents and validates engineered features. It does not rebuild the heavy full-data outputs.

## B1. `charged_share_2024_geo` - CRZ geography exposure

`charged_share_2024_geo` is already present in `hvfhv_monthly_panel.csv`. It is computed from 2024 geography rather than 2025 observed charges, so it is the correct main exposure for the HVFHV Model 2 design.

In [8]:
exposure_checks = pd.DataFrame({
    "check": [
        "panel rows",
        "unique zone-direction units",
        "missing charged_share_2024_geo rows",
        "minimum exposure",
        "maximum exposure",
    ],
    "value": [
        len(panel),
        panel[["zone", "direction"]].drop_duplicates().shape[0],
        int(panel["charged_share_2024_geo"].isna().sum()),
        round(float(panel["charged_share_2024_geo"].min()), 4),
        round(float(panel["charged_share_2024_geo"].max()), 4),
    ],
})
show(exposure_checks)

                              check  value
                         panel rows 5236.0
        unique zone-direction units  527.0
missing charged_share_2024_geo rows    3.0
                   minimum exposure    0.0
                   maximum exposure    1.0


## B2. `base_cost_ex_cbd` - DS_z denominator

For HVFHV, `base_cost_ex_cbd` is the rounded passenger cost excluding the CBD fee. It is the DS_z denominator after applying the $1 floor for burden-eligible trips.

In [9]:
denom = trips.copy()
calculated_base_cost = (denom["passenger_cost_pretip"] - denom["cbd_congestion_fee"]).round(2)
recorded_base_cost = pd.to_numeric(denom["base_cost_ex_cbd"], errors="coerce").round(2)
base_cost_diff = (calculated_base_cost - recorded_base_cost).abs()
print(f"Rows checked: {len(denom):,}")
print(f"Max absolute denominator difference: {base_cost_diff.max():.6f}")
print(f"Rows with difference > $0.01: {int((base_cost_diff > 0.01).sum()):,}")
print(f"Burden-analysis rows with base_cost_ex_cbd >= $1: {int((recorded_base_cost >= 1.0).sum()):,}")

Rows checked: 16,889
Max absolute denominator difference: 0.000000
Rows with difference > $0.01: 0
Burden-analysis rows with base_cost_ex_cbd >= $1: 16,889


## B3. Already produced upstream

The heavier outputs are produced upstream and reused here:

- `hvfhv_model1_zone_features.csv`: Model 1 zone-direction features and outcomes.
- `hvfhv_monthly_panel.csv`: Model 2 monthly zone-direction panel.
- `hvfhv_model2_exposure_validation.csv`: 2025 observed-fee diagnostic for the geography exposure.
- `hvfhv_pretrend_2024_diagnostic.csv` and `hvfhv_placebo_2023_2024_results.csv`: no-June pretrend/placebo diagnostics.

Model 3 remains postponed. Do not interpret the HVFHV Model 2 estimate as clean causal evidence because the 2023-vs-2024 placebo is also negative and similar in magnitude.